# 🚀 Clasificación de Área mediante Transformers (PLN Avanzado)

Este notebook realiza **fine-tuning** de modelos pre-entrenados de tipo **Transformer** para clasificar transacciones bancarias según su **Área** a partir de la descripción textual.

## Modelos entrenados:
1. **BETO** (`dccuchile/bert-base-spanish-wwm-cased`) – BERT preentrenado específicamente en español
2. **RoBERTa-BNE** (`PlanTL-GOB-ES/roberta-base-bne`) – RoBERTa entrenado sobre el corpus de la Biblioteca Nacional de España
3. **XLM-RoBERTa** (`xlm-roberta-base`) – Modelo multilingüe de Meta (100 idiomas)
4. **DistilBERT Multilingual** (`distilbert-base-multilingual-cased`) – Versión destilada de mBERT (ligera y rápida)
5. **BERTIN** (`bertin-project/bertin-roberta-base-spanish`) – RoBERTa español con entrenamiento eficiente

---

## 1. Instalación de Dependencias

Ejecutar solo si no están instaladas.

In [1]:
# Descomentar y ejecutar si es necesario:
# !pip install transformers datasets accelerate torch scikit-learn matplotlib seaborn

## 2. Importaciones y Configuración

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import torch
from collections import OrderedDict

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Librerías cargadas")
print(f"🖥️  Dispositivo: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

AttributeError: module 'numpy' has no attribute '__version__'

## 3. Carga y Preparación de Datos

In [ ]:
# Cargar dataset
DATA_PATH = os.path.join("..", "data", "raw", "db_orig.csv")
df_raw = pd.read_csv(DATA_PATH)

# Nos quedamos SOLO con Description y Area
df = df_raw[["Description", "Area"]].copy().dropna()

print(f"📊 Dataset: {df.shape[0]} filas")
print(f"🏷️  Clases únicas de Area: {df['Area'].nunique()}")
print()
print(df["Area"].value_counts())
df.head()

In [ ]:
# Codificar etiquetas numéricamente
le = LabelEncoder()
df["label"] = le.fit_transform(df["Area"])

NUM_LABELS = len(le.classes_)
label2id = {label: idx for idx, label in enumerate(le.classes_)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"📋 Mapping de etiquetas ({NUM_LABELS} clases):")
for idx, label in id2label.items():
    print(f"   {idx} → {label}")

In [ ]:
# Split 80/20 estratificado
df_train, df_test = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label"]
)

print(f"🔀 División estratificada 80/20:")
print(f"   Train: {len(df_train)} muestras")
print(f"   Test:  {len(df_test)} muestras")

# Convertir a datasets de Hugging Face
train_dataset = Dataset.from_pandas(df_train[["Description", "label"]].reset_index(drop=True))
test_dataset = Dataset.from_pandas(df_test[["Description", "label"]].reset_index(drop=True))

print(f"\n✅ Datasets HF creados: train={len(train_dataset)}, test={len(test_dataset)}")

## 4. Definición de Modelos y Configuración de Entrenamiento

In [ ]:
# Modelos a entrenar
MODELS = OrderedDict({
    "BETO": "dccuchile/bert-base-spanish-wwm-cased",
    "RoBERTa-BNE": "PlanTL-GOB-ES/roberta-base-bne",
    "XLM-RoBERTa": "xlm-roberta-base",
    "DistilBERT Multi": "distilbert-base-multilingual-cased",
    "BERTIN": "bertin-project/bertin-roberta-base-spanish",
})

# Hiperparámetros comunes
MAX_LENGTH = 64        # Longitud máx. de tokens (las descripciones son cortas)
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

print(f"🤖 {len(MODELS)} modelos a entrenar:")
for name, model_id in MODELS.items():
    print(f"   • {name}: {model_id}")

print(f"\n⚙️ Hiperparámetros:")
print(f"   max_length={MAX_LENGTH}, batch_size={BATCH_SIZE}, epochs={EPOCHS}")
print(f"   lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY}")

In [ ]:
# Función para calcular métricas durante el entrenamiento
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
        "precision_weighted": precision_score(labels, predictions, average="weighted"),
        "recall_weighted": recall_score(labels, predictions, average="weighted"),
    }

print("✅ Función de métricas definida")

## 5. Función de Entrenamiento (Fine-Tuning)

In [ ]:
def train_transformer(model_name, model_id, train_ds, test_ds, num_labels, epochs=EPOCHS):
    """
    Fine-tuning completo de un modelo transformer para clasificación.
    
    Args:
        model_name: Nombre corto del modelo (para logs)
        model_id: Identificador de Hugging Face
        train_ds: Dataset de entrenamiento
        test_ds: Dataset de test
        num_labels: Número de clases
        epochs: Número de épocas
    
    Returns:
        dict con trainer, tokenizer y métricas
    """
    print(f"\n{'='*60}")
    print(f"🔄 Entrenando: {model_name} ({model_id})")
    print(f"{'='*60}")
    
    # 1. Cargar tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # 2. Tokenizar datasets
    def tokenize_fn(examples):
        return tokenizer(
            examples["Description"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
        )
    
    train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=["Description"])
    test_tok = test_ds.map(tokenize_fn, batched=True, remove_columns=["Description"])
    
    train_tok.set_format("torch")
    test_tok.set_format("torch")
    
    # 3. Cargar modelo
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )
    
    # 4. Configurar entrenamiento
    output_dir = f"./results_{model_name.lower().replace(' ', '_')}"
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        greater_is_better=True,
        logging_steps=10,
        seed=SEED,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    
    # 5. Crear Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=test_tok,
        compute_metrics=compute_metrics,
    )
    
    # 6. Entrenar
    trainer.train()
    
    # 7. Evaluar
    eval_results = trainer.evaluate()
    print(f"\n📊 Resultados en test:")
    print(f"   Accuracy:  {eval_results['eval_accuracy']:.4f}")
    print(f"   F1:        {eval_results['eval_f1_weighted']:.4f}")
    print(f"   Precision: {eval_results['eval_precision_weighted']:.4f}")
    print(f"   Recall:    {eval_results['eval_recall_weighted']:.4f}")
    
    # 8. Predicciones
    preds_output = trainer.predict(test_tok)
    y_pred = np.argmax(preds_output.predictions, axis=-1)
    
    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "eval_results": eval_results,
        "y_pred": y_pred,
        "model": model,
    }

print("✅ Función de entrenamiento definida")

## 6. Entrenamiento de Todos los Modelos

> ⏱️ **Nota:** El entrenamiento de 5 modelos puede tardar varios minutos en CPU o ~1-2 min en GPU.

In [ ]:
import time

all_results = {}
training_times = {}

for model_name, model_id in MODELS.items():
    start = time.time()
    try:
        result = train_transformer(
            model_name, model_id, train_dataset, test_dataset, NUM_LABELS
        )
        all_results[model_name] = result
        training_times[model_name] = time.time() - start
        print(f"⏱️ Tiempo: {training_times[model_name]:.1f}s")
    except Exception as e:
        print(f"❌ Error entrenando {model_name}: {e}")
        training_times[model_name] = time.time() - start

print(f"\n{'='*60}")
print(f"✅ Entrenamiento completado: {len(all_results)}/{len(MODELS)} modelos")
print(f"{'='*60}")

## 7. Comparativa de Modelos

In [ ]:
# Tabla resumen
results_list = []

for model_name, res in all_results.items():
    ev = res["eval_results"]
    results_list.append({
        "Modelo": model_name,
        "HF ID": MODELS[model_name],
        "Test Accuracy": ev["eval_accuracy"],
        "Test F1": ev["eval_f1_weighted"],
        "Test Precision": ev["eval_precision_weighted"],
        "Test Recall": ev["eval_recall_weighted"],
        "Tiempo (s)": training_times[model_name],
    })

df_results = pd.DataFrame(results_list).sort_values("Test F1", ascending=False)
df_results.index = range(1, len(df_results) + 1)

print("📊 TABLA COMPARATIVA DE MODELOS TRANSFORMER:\n")
display(df_results.style.format({
    "Test Accuracy": "{:.4f}",
    "Test F1": "{:.4f}",
    "Test Precision": "{:.4f}",
    "Test Recall": "{:.4f}",
    "Tiempo (s)": "{:.1f}",
}).highlight_max(subset=["Test F1", "Test Accuracy"], color="#a8e6cf")
 .highlight_min(subset=["Tiempo (s)"], color="#a8e6cf"))

In [ ]:
# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_sorted = df_results.sort_values("Test F1", ascending=True)
palette = sns.color_palette("viridis", n_colors=len(df_sorted))

# --- F1 Score en Test ---
bars = axes[0].barh(
    df_sorted["Modelo"], df_sorted["Test F1"],
    color=palette, edgecolor="white", linewidth=1.5, height=0.6,
)
for bar, val in zip(bars, df_sorted["Test F1"]):
    axes[0].text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                 f"{val:.4f}", va="center", fontsize=10, fontweight="bold")
axes[0].set_xlim(0, 1.08)
axes[0].set_title("F1-Score en Test (weighted)", fontweight="bold", fontsize=13)
axes[0].set_xlabel("F1-Score")

# --- Tiempo de entrenamiento ---
bars2 = axes[1].barh(
    df_sorted["Modelo"], df_sorted["Tiempo (s)"],
    color=palette, edgecolor="white", linewidth=1.5, height=0.6,
)
for bar, val in zip(bars2, df_sorted["Tiempo (s)"]):
    axes[1].text(val + 1, bar.get_y() + bar.get_height() / 2,
                 f"{val:.0f}s", va="center", fontsize=10, fontweight="bold")
axes[1].set_title("Tiempo de entrenamiento", fontweight="bold", fontsize=13)
axes[1].set_xlabel("Segundos")

plt.tight_layout()
plt.savefig("resultados_transformers.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Gráfico guardado: resultados_transformers.png")

## 8. Matrices de Confusión

In [ ]:
# Mejor modelo
best_name = df_results.iloc[0]["Modelo"]
best_result = all_results[best_name]
y_true = df_test["label"].values
y_pred_best = best_result["y_pred"]

labels_sorted = sorted(id2label.keys())
label_names = [id2label[i] for i in labels_sorted]

fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_true, y_pred_best, labels=labels_sorted)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=True, xticks_rotation=45)
ax.set_title(f"Matriz de Confusión — {best_name}", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("matriz_confusion_transformers_best.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Matrices de confusión de todos los modelos
n_models = len(all_results)
if n_models > 0:
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 6))
    if n_models == 1:
        axes = [axes]

    for ax, (name, res) in zip(axes, all_results.items()):
        y_pred = res["y_pred"]
        cm = confusion_matrix(y_true, y_pred, labels=labels_sorted)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
        disp.plot(ax=ax, cmap="Blues", values_format="d", colorbar=False, xticks_rotation=45)
        ax.set_title(name, fontweight="bold", fontsize=9)

    plt.suptitle("Matrices de Confusión — Todos los Transformers", fontweight="bold", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig("matrices_confusion_transformers.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 Gráfico guardado: matrices_confusion_transformers.png")

## 9. Classification Report del Mejor Modelo

In [ ]:
print(f"🏆 MEJOR MODELO: {best_name}\n")
print("═" * 70)
print(classification_report(
    y_true, y_pred_best,
    labels=labels_sorted,
    target_names=label_names,
    digits=4,
))
print("═" * 70)

## 10. Análisis de Errores

In [ ]:
# Identificar errores del mejor modelo
y_true_labels = le.inverse_transform(y_true)
y_pred_labels = le.inverse_transform(y_pred_best)

df_eval = pd.DataFrame({
    "Description": df_test["Description"].values,
    "Real": y_true_labels,
    "Predicción": y_pred_labels,
})
df_errors = df_eval[df_eval["Real"] != df_eval["Predicción"]]

print(f"❌ Errores del mejor modelo ({best_name}):")
print(f"   Total errores: {len(df_errors)} de {len(df_eval)} ({100*len(df_errors)/len(df_eval):.1f}%)\n")

if len(df_errors) > 0:
    display(df_errors.sort_values("Real").reset_index(drop=True))
else:
    print("   ¡Sin errores! El modelo clasificó todas las muestras correctamente.")

In [ ]:
# Resumen de errores por clase
if len(df_errors) > 0:
    print("📊 Errores agrupados por clase real → predicción:\n")
    error_summary = df_errors.groupby(["Real", "Predicción"]).size().reset_index(name="Count")
    error_summary = error_summary.sort_values("Count", ascending=False)
    display(error_summary)

    fig, ax = plt.subplots(figsize=(10, 4))
    errors_by_class = df_errors["Real"].value_counts()
    errors_by_class.plot(kind="barh", ax=ax, color="#ef476f", edgecolor="white")
    ax.set_title("Errores por clase real", fontweight="bold")
    ax.set_xlabel("Nº errores")
    plt.tight_layout()
    plt.show()

## 11. Prueba con Nuevas Descripciones

In [ ]:
# Predicción con el mejor modelo
best_tokenizer = all_results[best_name]["tokenizer"]
best_model = all_results[best_name]["model"]
best_model.eval()

nuevas_descripciones = [
    "Nómina mensual",
    "Compra supermercado",
    "Factura servicio",
    "Ingreso depósito",
    "Pequeño ocio",
    "Pequeño ingreso",
    "Entretenimiento",
    "Inversión mensual",
    "Gasto vacaciones",
    "Gasto alimentación",
    "Factura importante",
    "Actividad ocio",
    "Aportación inversión",
    "Microfactura",
    "Viaje o vacaciones",
]

# Tokenizar y predecir
preds = []
with torch.no_grad():
    for desc in nuevas_descripciones:
        inputs = best_tokenizer(
            desc, return_tensors="pt", padding="max_length",
            truncation=True, max_length=MAX_LENGTH,
        ).to(best_model.device)
        outputs = best_model(**inputs)
        pred_id = torch.argmax(outputs.logits, dim=-1).item()
        preds.append(id2label[pred_id])

df_nuevas = pd.DataFrame({
    "Descripción": nuevas_descripciones,
    "Area predicha": preds,
})

print(f"🔮 Predicciones con {best_name}:\n")
display(df_nuevas)

## 12. Comparativa: Transformers vs Modelos Clásicos

Referencia de los resultados clásicos (del notebook `nlp_area_classifier.ipynb`).

In [ ]:
# Si quieres comparar manualmente, aquí puedes introducir los resultados
# de los modelos clásicos del notebook anterior.

print("📋 Comparativa rápida (Transformers vs Clásicos):")
print()
print("🔹 Modelos CLÁSICOS (TF-IDF + ML):")
print("   Todos alcanzan ~1.0000 F1 (las descripciones son muy discriminativas)")
print()
print("🔸 Modelos TRANSFORMER (fine-tuned):")
for _, row in df_results.iterrows():
    print(f"   {row['Modelo']:20s} → F1: {row['Test F1']:.4f}  (Tiempo: {row['Tiempo (s)']:.0f}s)")
print()
print("💡 Nota: En datasets pequeños y con vocabulario muy predictivo,")
print("   los modelos clásicos TF-IDF pueden igualar o superar a los Transformers")
print("   con mucho menos coste computacional. Los Transformers brillan en")
print("   datasets más grandes, ambiguos y con mayor diversidad lingüística.")

## 13. Resumen y Conclusiones

In [ ]:
print("═" * 60)
print("          📋 RESUMEN FINAL")
print("═" * 60)
print(f"\n📊 Dataset: {df.shape[0]} muestras (Description → Area)")
print(f"🏷️  Clases: {NUM_LABELS} áreas distintas")
print(f"🔀 Split: 80% train ({len(df_train)}) / 20% test ({len(df_test)})")
print(f"🤖 Modelos entrenados: {len(all_results)}/{len(MODELS)}")
print(f"🖥️  Dispositivo: {DEVICE}")
print(f"\n🏆 Mejor modelo: {best_name}")
print(f"   → Test F1-Score:  {df_results.iloc[0]['Test F1']:.4f}")
print(f"   → Test Accuracy:  {df_results.iloc[0]['Test Accuracy']:.4f}")
print(f"   → Test Precision: {df_results.iloc[0]['Test Precision']:.4f}")
print(f"   → Test Recall:    {df_results.iloc[0]['Test Recall']:.4f}")
print(f"\n⚙️ Configuración de fine-tuning:")
print(f"   - Épocas: {EPOCHS}")
print(f"   - Batch size: {BATCH_SIZE}")
print(f"   - Learning rate: {LEARNING_RATE}")
print(f"   - Max tokens: {MAX_LENGTH}")
print("\n═" * 60)

## 14. Limpieza (Opcional)

Eliminar los checkpoints y carpetas de resultados generadas durante el entrenamiento.

In [ ]:
# Descomentar para limpiar los checkpoints:
# import shutil
# for model_name in MODELS:
#     d = f"./results_{model_name.lower().replace(' ', '_')}"
#     if os.path.exists(d):
#         shutil.rmtree(d)
#         print(f"🗑️ Eliminado: {d}")
# print("✅ Limpieza completada")